<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14.%20file_uploads_and_static_assets.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 14: File Uploads</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 16: Testing &#8594;</a>
</div>


# Chapter 15: Error Handling and Logging -- Graceful Failures

> *"Every app crashes sometimes. The question is: does it show a cryptic 500 page, or a friendly error page? And do YOU know about the crash? Error handling + logging is the difference between a professional app and an amateur one."*


## 🎯 The Big Picture

When Flask encounters a problem two things must happen simultaneously:

1. **The user** sees a friendly, on-brand error page -- no technical details exposed.
2. **You (the developer)** immediately know about the error with enough context to fix it.

Without proper error handling, users see raw Python tracebacks (a security risk). Without logging you fly blind -- you won't know the app is failing until a user emails you.

```
User hits broken route
         |
         v
   Flask error handler  -->  renders 404.html  -->  User sees friendly page
         |
         v
   app.logger.error()   -->  log file / Sentry  -->  Developer gets alerted
```

This chapter covers both pillars: **custom error pages** and **structured logging**.


## 🧠 Core Concepts: The "Why"

The easiest way to read this section is to keep one plain-language question in view: what is The "Why" actually responsible for? Once that job is clear, the terminology stops feeling arbitrary and the details start to line up.

### The Two Audiences for Errors

Think of a hospital emergency room:
- **Patients** hear: *"We're taking care of you."* -- reassuring, no jargon.
- **Doctors** get: *"SpO2 at 85%, administer O2 immediately"* -- detailed, actionable.

| Audience | What they see | What they need |
|----------|---------------|----------------|
| **Users** | Friendly HTML error page | Reassurance + what to do next |
| **Developers** | Full traceback in logs | Enough detail to reproduce and fix |

### The Golden Rule

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

> **Never show a Python traceback to a user in production. Never.**

A traceback reveals your file system structure, library versions (useful for attackers), and potentially sensitive config values.
### HTTP Error Codes

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

| Code | Meaning | When it happens |
|------|---------|-----------------|
| `400` | Bad Request | Malformed input |
| `401` | Unauthorized | Not logged in |
| `403` | Forbidden | Logged in but not allowed |
| `404` | Not Found | Wrong URL |
| `429` | Too Many Requests | Rate limited |
| `500` | Internal Server Error | Your code crashed |


## ⚡ Syntax & First Usage

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

### Registering Custom Error Handlers

The `@app.errorhandler()` decorator intercepts errors before Flask returns defaults:

> ❓ **What is a decorator?** A decorator is a function that wraps another function to add behaviour before or after it runs. `@app.route('/')` is shorthand for `index = app.route('/')(index)` — it registers your view function with Flask's URL map without any explicit call.

In [ ]:

from flask import Flask, render_template_string

app = Flask(__name__)

@app.errorhandler(404)
def not_found_error(error):
    html = "<h1>404 - Page Not Found</h1><p>The page you requested doesn't exist.</p>"
    return render_template_string(html), 404   # status code is the second return value

@app.errorhandler(500)
def internal_error(error):
    # db.session.rollback()  # critical in a real app -- resets broken DB state
    html = "<h1>500 - Something Went Wrong</h1><p>We are looking into it!</p>"
    return render_template_string(html), 500

print("Error handlers registered!")
print()
print("Key points:")
print("  1. The handler receives the error object as its argument")
print("  2. The HTTP status code MUST be the second return value")
print("  3. Without handlers, Flask returns default plain-text error pages")


### Using Real Templates (Blueprint Pattern)

A helpful beginner shortcut here is to ask why the system is split this way at all. Once the separation of responsibilities is clear, the patterns feel like practical decisions instead of abstract rules.



In [ ]:

# app/errors/__init__.py  -- Blueprint-based error handling (best practice)

code_blueprint = """
from flask import Blueprint, render_template
from app import db

errors = Blueprint("errors", __name__)

@errors.app_errorhandler(404)
def not_found_error(error):
    return render_template("errors/404.html"), 404

@errors.app_errorhandler(403)
def forbidden_error(error):
    return render_template("errors/403.html"), 403

@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()          # ALWAYS reset broken DB state
    return render_template("errors/500.html"), 500
"""

# templates/errors/404.html
template_404 = """
{% extends "base.html" %}
{% block title %}Page Not Found{% endblock %}
{% block content %}
  <div class="error-container text-center py-5">
    <h1 class="display-1 text-muted">404</h1>
    <h2>Oops - that page does not exist</h2>
    <p class="text-muted">The page you requested could not be found.</p>
    <a href="{{ url_for('main.index') }}" class="btn btn-primary mt-3">Go Home</a>
  </div>
{% endblock %}
"""

print("Blueprint error handler pattern:")
print(code_blueprint)
print("templates/errors/404.html:")
print(template_404)


## 🔬 Deep Dive

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

### Custom HTTP Exceptions

Define named exceptions that carry semantic meaning:

In [ ]:

from werkzeug.exceptions import HTTPException

class PostNotFound(HTTPException):
    code = 404
    description = "The blog post does not exist or has been deleted."

class InsufficientPermissions(HTTPException):
    code = 403
    description = "You do not have permission to perform this action."

class RateLimitExceeded(HTTPException):
    code = 429
    description = "Too many requests. Please try again in a minute."

example_route = """
@app.route("/post/<int:post_id>")
def view_post(post_id):
    post = Post.query.get(post_id)
    if post is None:
        raise PostNotFound()                    # descriptive, self-documenting
    if not post.is_published and post.author != current_user:
        raise InsufficientPermissions()
    return render_template("post.html", post=post)
"""

print("Custom exceptions:")
for exc in [PostNotFound, InsufficientPermissions, RateLimitExceeded]:
    print(f"  {exc.__name__:<30} HTTP {exc.code}")
print()
print("Usage in routes:")
print(example_route)
print("Benefits: self-documenting, single place to update, testable with try/except.")


### `abort()` vs Raising Exceptions vs `.get_or_404()`

The hard part here is usually not the syntax but the boundary between similar ideas. Keep comparing the job each option does best, the tradeoff it introduces, and the clues that tell you which one fits the situation in front of you.



In [ ]:

# Three ways to trigger error responses

abort_example = """
# abort() -- best for simple inline one-liners
from flask import abort

@app.route("/admin/")
def admin_dashboard():
    if not current_user.is_admin:
        abort(403)
    return render_template("admin/dashboard.html")
"""

raise_example = """
# raise CustomException -- best for semantic clarity
@app.route("/post/<int:id>/delete", methods=["POST"])
def delete_post(id):
    post = Post.query.get_or_404(id)
    if post.author_id != current_user.id:
        raise InsufficientPermissions()    # clear, self-documenting
    db.session.delete(post)
    db.session.commit()
    return redirect(url_for("main.index"))
"""

shortcut_example = """
# get_or_404() -- SQLAlchemy convenience shortcuts
post = Post.query.get_or_404(post_id)
user = User.query.filter_by(username=name).first_or_404()
# Automatically abort(404) if not found; return object if found.
"""

print("abort()      -- immediate, one-liner for simple cases")
print("raise        -- named exceptions for semantic clarity")
print("get_or_404() -- best shortcut when fetching by primary key")
print()
for name, code in [("abort()", abort_example), ("raise", raise_example), ("shortcut", shortcut_example)]:
    print(f"=== {name} ===")
    print(code)


### Flask's Built-in Logger

Flask exposes Python's logging module as `app.logger`:

In [ ]:

log_levels = {
    "DEBUG":    10,   # Granular debugging -- suppressed in production
    "INFO":     20,   # Normal operational events
    "WARNING":  30,   # Unexpected but handled
    "ERROR":    40,   # Serious problem, some functionality lost
    "CRITICAL": 50,   # System-wide failure
}

print("Log level hierarchy (lowest to highest severity):")
print("=" * 50)
for level, value in log_levels.items():
    bar = "█" * (value // 5)
    print(f"  {level:<10} ({value:2d})  {bar}")

usage = """
# Inside routes (use current_app.logger in blueprints)
app.logger.debug("Detailed debug -- shown only in development")
app.logger.info("User 42 logged in successfully")
app.logger.warning("Disk usage at 90% -- check storage")
app.logger.error("Payment timeout for order 99", exc_info=True)
app.logger.critical("Database pool exhausted")
"""
print()
print("Usage:", usage)
print("Setting level to INFO suppresses all DEBUG messages.")
print("Setting level to WARNING suppresses INFO and DEBUG.")


### Setting Up File Logging with `RotatingFileHandler`

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.



In [ ]:

import logging
from logging.handlers import RotatingFileHandler

logging_setup = """
def configure_logging(app):
    if not app.debug and not app.testing:
        import os
        if not os.path.exists("logs"):
            os.mkdir("logs")

        handler = RotatingFileHandler(
            "logs/app.log",
            maxBytes=10 * 1024 * 1024,  # 10 MB per file
            backupCount=10              # app.log + app.log.1 ... .10 = 110 MB max
        )
        formatter = logging.Formatter(
            "%(asctime)s %(levelname)s: %(message)s [in %(pathname)s:%(lineno)d]"
        )
        handler.setFormatter(formatter)
        handler.setLevel(logging.INFO)

        app.logger.addHandler(handler)
        app.logger.setLevel(logging.INFO)
        app.logger.info("Application startup")
"""

print("Production logging configuration:")
print(logging_setup)
print("Example log output:")
print("  2024-01-15 14:23:01 INFO: Application startup [in app/__init__.py:45]")
print("  2024-01-15 14:23:05 INFO: User 42 logged in [in app/auth/routes.py:87]")
print("  2024-01-15 14:25:10 WARNING: Failed login for test@ex.com [in routes.py:102]")
print("  2024-01-15 14:30:22 ERROR: 500 Internal Error [in app/errors/handlers.py:18]")


### Logging Throughout Your Application

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:

route_logging = """
from flask import current_app

# Authentication -- always log these events
@auth.route("/login", methods=["POST"])
def login():
    user = User.query.filter_by(email=form.email.data).first()
    if user and user.check_password(form.password.data):
        login_user(user)
        current_app.logger.info(f"User {user.id} logged in")
        return redirect(url_for("main.index"))

    current_app.logger.warning(
        f"Failed login: {form.email.data} from {request.remote_addr}"
    )
    return render_template("auth/login.html")


# Destructive operations -- log with user ID and resource ID
@posts.route("/post/<int:id>/delete", methods=["POST"])
@login_required
def delete_post(id):
    post = Post.query.get_or_404(id)
    current_app.logger.info(f"Post {post.id} deleted by user {current_user.id}")
    db.session.delete(post)
    db.session.commit()
    return redirect(url_for("main.index"))


# Error handler -- log with full traceback
@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()
    current_app.logger.error(f"500 error: {error}", exc_info=True)
    return render_template("errors/500.html"), 500
"""

print("What to log:")
print("  ALWAYS: Authentication events (logins, failures, logouts)")
print("  ALWAYS: Destructive actions (deletes, bulk updates)")
print("  ALWAYS: External service calls (email, payment, API)")
print("  ALWAYS: All 500 errors with exc_info=True for full traceback")
print()
print("NEVER log: passwords, API keys, tokens, credit card numbers, SSNs")
print()
print(route_logging)


### Debug Mode vs Production Mode

The hard part here is usually not the syntax but the boundary between similar ideas. Keep comparing the job each option does best, the tradeoff it introduces, and the clues that tell you which one fits the situation in front of you.



In [ ]:

comparison = """
+----------------------+-------------------------------+------------------------------+
| Feature              | DEBUG=True (dev only!)        | DEBUG=False (production)     |
+----------------------+-------------------------------+------------------------------+
| Exception display    | Werkzeug interactive debugger | Custom error handler runs    |
| Auto-reload          | Yes, on code changes          | No                           |
| Error traceback      | Shown in browser              | Logged only, never shown     |
| Remote code exec     | Possible via debugger         | N/A                          |
| Performance          | Slower                        | Faster                       |
| Security             | NEVER use in production       | Safe                         |
+----------------------+-------------------------------+------------------------------+
"""

config_code = """
class ProductionConfig:
    DEBUG   = False   # CRITICAL: never True in production
    TESTING = False
    SECRET_KEY = os.environ["SECRET_KEY"]
    SQLALCHEMY_DATABASE_URI = os.environ["DATABASE_URL"]

class DevelopmentConfig:
    DEBUG   = True    # interactive debugger, auto-reload

class TestingConfig:
    DEBUG   = False
    TESTING = True    # disables error catching for cleaner test assertions
    SQLALCHEMY_DATABASE_URI = "sqlite:///:memory:"
    WTF_CSRF_ENABLED = False
"""

print(comparison)
print("Why DEBUG=True in production is a CRITICAL vulnerability:")
print("  Werkzeug debugger allows REMOTE CODE EXECUTION via browser")
print("  Exposes full source code, file paths, config values")
print("  Attackers actively scan for Flask debug mode on the internet")
print()
print("Config pattern:")
print(config_code)


### ⚖️ Comparison: `print()` vs `app.logger`

The hard part here is usually not the syntax but the boundary between similar ideas. Keep comparing the job each option does best, the tradeoff it introduces, and the clues that tell you which one fits the situation in front of you.



In [ ]:

comparison = """
+----------------------+---------------------------+------------------------------+
| Feature              | print()                   | app.logger                   |
+----------------------+---------------------------+------------------------------+
| Output destination   | stdout only               | Console + files + Sentry +…  |
| Severity levels      | None (all identical)      | DEBUG/INFO/WARNING/ERROR     |
| Suppress by level    | Must delete manually      | setLevel() in config         |
| Timestamps           | No                        | Yes, via Formatter           |
| Module + line number | No                        | Yes, via Formatter           |
| Production-safe      | Clutters stdout           | Fully configurable           |
| Integration          | None                      | Sentry, ELK, CloudWatch, …   |
+----------------------+---------------------------+------------------------------+
"""

bad = """
# BAD -- print() in production
@app.route("/login", methods=["POST"])
def login():
    print(f"DEBUG: login for {request.form['email']}")   # no timestamp
    print(f"DEBUG: password: {request.form['password']}") # SECURITY NIGHTMARE
"""

good = """
# GOOD -- structured logging
@app.route("/login", methods=["POST"])
def login():
    current_app.logger.debug("Login form submitted")          # filtered in prod
    if not user:
        current_app.logger.warning(f"Failed: {email[:3]}***") # partial only
"""

print(comparison)
print("BAD (print):", bad)
print("GOOD (logger):", good)


### Sentry: Production Error Monitoring

Performance advice only becomes useful once you tie each technique to the bottleneck it addresses. Keep asking what is slow, how you can prove it, and what side effect each optimization introduces.



In [ ]:

sentry_setup = """
# 1. Install: pip install sentry-sdk[flask]

# 2. Initialize in create_app() BEFORE registering routes:
import sentry_sdk
from sentry_sdk.integrations.flask import FlaskIntegration

def create_app(config_name="default"):
    app = Flask(__name__)

    if not app.debug and not app.testing:
        sentry_sdk.init(
            dsn=os.environ.get("SENTRY_DSN"),       # never hardcode!
            integrations=[FlaskIntegration()],
            traces_sample_rate=0.1,                 # 10% of requests for perf
            environment="production",
            release=os.environ.get("GIT_SHA")       # link errors to deploys
        )

    return app
"""

features = [
    "Full Python tracebacks with local variable values at each stack frame",
    "Request context (URL, method, headers, POST data)",
    "User context (configure with sentry_sdk.set_user())",
    "Release tracking -- know which deploy introduced a bug",
    "Email/Slack/PagerDuty alerts for new issues",
    "Smart grouping -- same error counted, not duplicated",
    "Performance tracing -- slow routes and N+1 queries",
]

print("Sentry setup:")
print(sentry_setup)
print("What Sentry captures automatically:")
for f in features:
    print(f"  OK: {f}")
print()
print("Free tier: ~5,000 errors/month. Perfect for most small-to-medium projects.")


## 🧪 What If?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

### What if `db.session.rollback()` is missing in the 500 handler?

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.



In [ ]:

explanation = """
WITHOUT rollback() in the 500 handler:
---------------------------------------
Request 1: POST /create-post
  -> db.session.add(new_post)       <-- transaction started
  -> notify_followers()             <-- raises SMTPException!
  -> 500 handler fires              <-- session NOT cleaned up
  -> DB session left in invalid state

Request 2 (same Gunicorn worker): GET /my-posts
  -> Post.query.all()
     sqlalchemy.exc.InvalidRequestError:
     "Can't reconnect until invalid transaction is rolled back"
  -> User sees ANOTHER 500! Cascading failure.

WITH rollback() in the 500 handler:
--------------------------------------
Request 1: POST /create-post
  -> db.session.add(new_post)
  -> notify_followers()             <-- raises SMTPException!
  -> 500 handler: db.session.rollback()  <-- session cleaned up
  -> Friendly error page shown

Request 2: Works perfectly.
"""

handler = """
@errors.app_errorhandler(500)
def internal_error(error):
    db.session.rollback()    # non-negotiable
    current_app.logger.error(f"500 error: {error}", exc_info=True)
    return render_template("errors/500.html"), 500
"""

print(explanation)
print("Correct 500 handler:")
print(handler)


### What if you log sensitive data? / What if logs grow forever?

A useful beginner mental model here is to separate the shape of the data from the operations performed on it. Once you know what is being represented and who depends on that representation, the rules become easier to predict.



In [ ]:

print("=== Sensitive Data in Logs ===")
print()
dangerous = """
# CATASTROPHICALLY DANGEROUS
app.logger.info(f"Login: email={email}, password={password}")
# If this log file leaks: every user's password is exposed.
# This has caused real company-ending security breaches.
"""
safe = """
# SAFE -- log only non-sensitive identifiers
app.logger.info(f"Successful login: user_id={user.id}")
app.logger.warning(f"Failed login: {email[:3]}***")
"""
print("DANGEROUS:", dangerous)
print("SAFE:", safe)
print("Never log: passwords, tokens, credit cards, SSNs, OAuth tokens")

print()
print("=== Log File Growth ===")
print()
import math
print(f"{'Scenario':<30} {'MB/day':>10} {'GB/year':>10}")
print("-" * 52)
for reqs_per_sec, kb_per_req in [(1, 1), (10, 1), (100, 1), (100, 5)]:
    mb_per_day = reqs_per_sec * 86400 * kb_per_req / 1024
    gb_per_year = mb_per_day * 365 / 1024
    label = f"{reqs_per_sec} req/s, {kb_per_req}KB/req"
    print(f"  {label:<28} {mb_per_day:>10.1f} {gb_per_year:>10.1f}")

solution = """
from logging.handlers import RotatingFileHandler
handler = RotatingFileHandler(
    "logs/app.log",
    maxBytes=10 * 1024 * 1024,  # 10 MB per file
    backupCount=10               # max 110 MB total -- always bounded
)
"""
print()
print("Solution: RotatingFileHandler")
print(solution)


## 🚀 Real-World Project Link

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

In our **Blog Application**:

```
flask-blog/
├── app/
│   ├── errors/
│   │   ├── __init__.py         <- Blueprint definition
│   │   ├── handlers.py         <- @errorhandler + db.session.rollback()
│   │   └── templates/errors/
│   │       ├── 404.html        <- styled "Not Found" page
│   │       ├── 403.html        <- styled "Forbidden" page
│   │       └── 500.html        <- styled "Server Error" with apology
│   └── __init__.py             <- configure_logging() in create_app()
├── logs/
│   ├── .gitignore              <- IMPORTANT: exclude logs from git!
│   └── app.log                 <- RotatingFileHandler target
└── config.py                   <- per-environment DEBUG, TESTING, log level
```

**What gets logged:** every login/logout (INFO), post CRUD (INFO), failed auth + IP (WARNING), all 500 errors with traceback (ERROR). Sentry captures production exceptions with user context. Error pages match the blog brand.

> ❓ **Why split code into Blueprints?** A single file with hundreds of routes becomes unreadable. Blueprints group related routes (e.g., all auth routes) into pluggable mini-applications that you register on the main app — like separate chapters in a book.

## 📋 Chapter Summary & Cheat Sheet

### Custom Error Handlers

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

```python
@app.errorhandler(404)
def not_found(error):
    return render_template('errors/404.html'), 404

@app.errorhandler(500)
def server_error(error):
    db.session.rollback()   # ALWAYS rollback on 500
    return render_template('errors/500.html'), 500
```
### Custom Exceptions

A first read goes better when you connect every new idea to a concrete responsibility and a concrete use case. Once those anchors are in place, the terminology in this section becomes much easier to reuse accurately.

```python
from werkzeug.exceptions import HTTPException
class PostNotFound(HTTPException):
    code = 404
    description = "Post does not exist."
```
### Triggering Errors

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

```python
abort(404)                          # immediate inline
raise PostNotFound()                # named, semantic
post = Post.query.get_or_404(id)    # SQLAlchemy shortcut
```
### Production Logging

The important beginner shift here is moving from “it works on my machine” to “it behaves predictably in a real environment.” Each step matters because it reduces surprises when the software runs elsewhere.

```python
from logging.handlers import RotatingFileHandler
handler = RotatingFileHandler('logs/app.log', maxBytes=10_000_000, backupCount=10)
handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s: %(message)s'))
app.logger.addHandler(handler)
app.logger.setLevel(logging.INFO)
```
### Log Levels

If this topic is new, keep translating each new term into a simple question: what job does it do, when would you reach for it, and what would you confuse it with if you were moving too quickly? That habit makes the section easier to retain.

```python
app.logger.debug("dev detail")                    # filtered in production
app.logger.info("user 42 logged in")              # normal events
app.logger.warning("disk 90% full")               # investigate soon
app.logger.error("payment failed", exc_info=True) # serious + traceback
app.logger.critical("db unreachable")             # immediate action
```
### Security Rules

Security topics become much easier to follow when you separate the threat from the defense. As you read, keep asking what can go wrong, what protection addresses it, and what assumption that protection depends on.

| Rule | Why |
|------|-----|
| Never `DEBUG=True` in production | Remote code execution |
| Never log passwords/tokens | Log leaks expose all users |
| Always `db.session.rollback()` in 500 handler | Prevents cascading DB failures |
| Use `RotatingFileHandler` | Prevents disk-filling logs |
| Add Sentry in production | Real-time error alerts |


## 💪 Practice Prompts

**1. Custom Error Pages**
Create a Flask app with custom HTML handlers for 404, 403, and 500. The 404 page should include a search bar and home link. The 403 page should explain "forbidden" in plain English. The 500 page should include a contact email and apology. Test by visiting non-existent routes and calling `abort(403)`.

**2. Structured Logging**
Add production-ready logging to a Flask app. Configure `RotatingFileHandler` writing to `logs/app.log` (10 MB rotation, 10 backups). Add `logger.info()` on every successful POST, `logger.warning()` on failed logins, and `logger.error(exc_info=True)` in your 500 handler. Run the app, trigger events, then inspect the log file format.

**3. Custom Exception Classes**
Define 3+ custom `HTTPException` subclasses for a domain of your choice (e.g., `ArticleNotFound`, `CommentLocked`, `ProfilePrivate`). Register `app.errorhandler` for each. Replace all bare `abort()` calls in your routes with these named exceptions and notice how the code becomes more readable.


<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14.%20file_uploads_and_static_assets.ipynb' style='padding:6px 14px; background:#007bff; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#8592; Chapter 14: File Uploads</a>
  <a href='../TOC.md' style='padding:6px 14px; background:#6c757d; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>&#128218; Table of Contents</a>
  <a href='16.%20testing_flask_applications.ipynb' style='padding:6px 14px; background:#28a745; color:white; border-radius:4px; text-decoration:none; font-weight:bold;'>Chapter 16: Testing &#8594;</a>
</div>

<div style='width:100%; display:flex; justify-content:space-between; align-items:center; margin: 1em 0;'>
  <a href='14. file_uploads_and_static_assets.ipynb' style='font-weight:bold; font-size:1.05em;'>&larr; Previous</a>
  <a href='../TOC.md' style='font-weight:bold; font-size:1.05em; text-align:center;'>Table of Contents</a>
  <a href='16. testing_flask_applications.ipynb' style='font-weight:bold; font-size:1.05em;'>Next &rarr;</a>
</div>
